In [ ]:
# SPDX-FileCopyrightText: 2024-2026 Dan J. Bower <dbower@eaps.ethz.ch>
# SPDX-FileCopyrightText: 2024-2026 Paolo A. Sossi
#
# SPDX-License-Identifier: Apache-2.0

import logging
from pathlib import Path
from typing import Literal

import numpy as np

from atmodeller import ChemicalSpecies, EquilibriumModel, Planet, PurePhase, debug_logger, earth

logger = debug_logger()
logger.setLevel(logging.INFO)

# For more output use DEBUG
# logger.setLevel(logging.DEBUG)

# Atmosphere

This notebook is available at `notebooks/examples/atmosphere.ipynb` and is easiest to obtain by downloading the source code.

*Atmodeller* allows you to simultaneously solve for surface conditions and an atmospheric P-T profile under equilibrium assumptions. In this example, we follow the model presented in Hakim et al. (2026) for TOI-421b.

In [ ]:
# Mass of TOI-421b and the radius of its magma-envelope boundary (MEB), from the mass-radius
# relation of Hakim et al. (2018), Icarus, for an atmospheric radius of 2.64 Earth radii
planet_mass = 6.7 * earth.mass  # kg
MEB_radius = 1.65 * earth.radius  # metre

# Temperature of TOI-421b
surface_temperature = 3000  # K
top_temperature = 920  # K

# Whether to export the data to Excel and pickle files, and where
WRITE_OUTPUT = False
OUTPUT_DIRECTORY = Path("output") / "atmosphere"

# There are a few different output formats
output_format: Literal["elements_species", "named_arrays"] = "named_arrays"

if WRITE_OUTPUT:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

We set up arrays for pressure (in bar) and temperature (in K) profile. The "trick" is to use ``np.nan`` as a placeholder indicating that the surface pressure should be solved for based on the total volatile mass in the atmosphere, that is, by enforcing mechanical pressure balance at the surface. For all other entries where a pressure value is provided, that value is used directly. The temperature array aligns positionally with the pressure array: the first entry gives the surface temperature, and the last entry corresponds to the temperature at the final pressure level.

In [ ]:
# Note: outside the temperature range of their thermodynamic data, gas data are extrapolated and
# condensates are not stable
temperature = np.array([surface_temperature, 1500, 1000, top_temperature])
pressure = np.array([np.nan, 1, 1e-2, 1e-4])

We then create the model using the gaseous and condensed species we wish to include:

In [ ]:
# Gas species
H2O_g = ChemicalSpecies.create_gas("H2O")
H2_g = ChemicalSpecies.create_gas("H2")
O2_g = ChemicalSpecies.create_gas("O2")
OSi_g = ChemicalSpecies.create_gas("OSi")
H4Si_g = ChemicalSpecies.create_gas("H4Si")
CO2_g = ChemicalSpecies.create_gas("CO2")
CO_g = ChemicalSpecies.create_gas("CO")
CH4_g = ChemicalSpecies.create_gas("CH4")
N2_g = ChemicalSpecies.create_gas("N2")
NH3_g = ChemicalSpecies.create_gas("NH3")
He_g = ChemicalSpecies.create_gas("He")
gas_species = (H2O_g, H2_g, O2_g, OSi_g, H4Si_g, CO2_g, CO_g, CH4_g, N2_g, NH3_g, He_g)

# Condensates
# Note: a condensate is not stable outside the temperature range of its thermodynamic data.
# from_formula stitches the solids and liquid of a formula together in order of temperature, so
# silica (melting at 1996 K) and silicon (melting at 1690 K) are each one species.
O2Si_cd = PurePhase.from_formula("SiO2")
Si_cd = PurePhase.from_formula("Si")
N4Si3_cd = PurePhase.from_formula("N4Si3")
C_graphite = PurePhase.from_component("graphite")
CSi_silicon_carbide = PurePhase.from_component("silicon-carbide-beta")
condensates = (O2Si_cd, C_graphite, CSi_silicon_carbide, Si_cd, N4Si3_cd)

We set the mass constraints based on a previous *Atmodeller* calculation that accounted for real-gas behaviour and dissolution into a magma ocean. From that calculation, we extract elemental abundances in the gas phase only (i.e. the atmosphere) and use them as abundance constraints. The values below correspond to a model with solar metallicity and a fully molten mantle.

In [ ]:
mass_constraints = {
    "H": 5.73802837470845e22,
    "He": 7.13997e21,
    "C": 1.18962697880417e21,
    "N": 1.99427e17,
    "O": 1.94361960955633e22,
    "Si": 3.5648e23,
}

Since we are using the gas masses from a previous calculation directly (which already accounted for gas solubility), we set ``mantle_melt_fraction=0``. The specification of the planet's mass and surface radius is used only when computing the mechanical pressure balance&mdash;that is, for determining the first pressure entry. These parameters play no role when the pressure is prescribed directly.

In [ ]:
planet = Planet.from_species(
    gas_species,
    temperature=temperature,
    planet_mass=planet_mass,
    mantle_melt_fraction=0,
    surface_radius=MEB_radius,
    pressure=pressure,
    condensates=condensates,
)

Finally, we create and solve the model:

In [ ]:
# Create the model with mass constraints
model = EquilibriumModel.from_state(planet, mass_constraints=mass_constraints)

# Solve with an automatic initial guess and default solver settings
output = model.solve_with_default()

# Optionally print solver stats to the logger
output.solver_stats_to_logger()

# Quick look at the solution
# output.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output.to_excel(OUTPUT_DIRECTORY / "atmosphere", output_format=output_format)

    # Write the data to a pickle file with dataframes
    output.to_pickle(OUTPUT_DIRECTORY / "atmosphere", output_format=output_format)

The output will contain four model evaluations, and you can see that the gas pressure satisfies the prescribed constraints, with the first entry enforcing the mechanical pressure balance at the surface. This workflow therefore provides a convenient way to evaluate the atmospheric chemical signature at different temperatures and pressures while simultaneously calculating the conditions at the planetary surface.

In [ ]:
# Show the state
output.to_dataframes()["state"]